# 04 — TFT (Temporal Fusion Transformer)

Notebook ini berdiri sendiri: jalankan `make preprocess` sekali dulu
(datanya diambil dari BigQuery), lalu Run All di sini untuk train + evaluasi
+ upload model TFT saja.

Input-nya tabel `preprocessed_data_hourly` + `preprocessing_metadata_hourly`
hasil `make preprocess`: semua stasiun + kalender (sin/cos) untuk semua model,
ditambah akumulasi hujan DAS milik stasiun target. `USE_RAINFALL = False`
menjalankan ablation tanpa hujan; hasilnya disimpan sebagai `<model>_norain`.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

import io
import json
import pickle
import numpy as np
import pandas as pd

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

# False = ablation tanpa fitur hujan; hasil disimpan sebagai <model>_norain.
USE_RAINFALL = True

PREPROCESSED_TABLE = f"{PROJECT_ID}.{DATASET}.preprocessed_data_hourly"
METADATA_TABLE     = f"{PROJECT_ID}.{DATASET}.preprocessing_metadata_hourly"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# --- Metadata: kolom kalender/stasiun/fitur + mapping nama kolom BigQuery ---
meta_row = client.query(f"""
    SELECT payload
    FROM `{METADATA_TABLE}`
    ORDER BY saved_at DESC
    LIMIT 1
""").to_dataframe().iloc[0]
metadata = json.loads(meta_row["payload"])
station_cols  = metadata["station_cols"]
column_map    = metadata["column_map"]                 # station -> kolom BigQuery
reverse_column_map = {v: k for k, v in column_map.items()}

# Fitur: semua stasiun + kalender (sin/cos) untuk semua model, ditambah
# akumulasi hujan DAS milik stasiun target (lihat `make preprocess`).
if "base_feature_cols" not in metadata:
    raise RuntimeError("Metadata versi lama (tanpa fitur hujan/kalender sin-cos); jalankan `make preprocess` dulu.")
BASE_FEATURE_COLS    = metadata["base_feature_cols"]
RAIN_COLS_BY_STATION = metadata["rain_cols_by_station"] if USE_RAINFALL else {}
FEATURES_BY_STATION  = {s: BASE_FEATURE_COLS + RAIN_COLS_BY_STATION.get(s, []) for s in station_cols}
ALL_FEATURE_COLS     = BASE_FEATURE_COLS + [c for cols in RAIN_COLS_BY_STATION.values() for c in cols]


def features_for(target):
    """Kolom input model untuk stasiun target (urutan sama dengan serving)."""
    return FEATURES_BY_STATION[target]


# --- df_train / df_val / df_test dari tabel preprocessed_data --------------
def load_split(split):
    query = f"""
        SELECT *
        FROM `{PREPROCESSED_TABLE}`
        WHERE split = @split
        ORDER BY datetime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("split", "STRING", split),
    ])
    df = client.query(query, job_config=job_config).to_dataframe()
    df = df.set_index("datetime").rename(columns=reverse_column_map)
    return df[ALL_FEATURE_COLS]


df_train = load_split("train")
df_val   = load_split("val")
df_test  = load_split("test")

print(f"Stations : {station_cols}")
n_rain = {s: len(c) for s, c in RAIN_COLS_BY_STATION.items()}
print(f"Features : {len(BASE_FEATURE_COLS)} dasar + fitur hujan per stasiun: {n_rain} (USE_RAINFALL={USE_RAINFALL})")
print(f"Train/Val/Test: {len(df_train):,} / {len(df_val):,} / {len(df_test):,} rows")

## Konfigurasi BigQuery — Hasil Model

Tabel hasil untuk model ini (semua di dataset `tables`):

- `tft_val_metrics_hourly` — metrik validasi per (station, horizon, metric).
- `tft_weights_hourly` — bobot terlatih per (stasiun, horizon), disimpan sebagai `BYTES`.
- `tft_predictions_hourly` — prediksi test-set per (station, horizon, datetime).
- `tft_metrics_hourly` — metrik test-set per (station, horizon, metric).

Bobot ditulis ke BigQuery segera setelah training (bukan ke disk), sehingga
langkah test-set evaluation di bawah bisa memuat ulang bobot dari BigQuery
saja bila objek model tidak lagi ada di memori (mis. notebook dibuka ulang).

In [ ]:
MODEL_NAME = "tft" if USE_RAINFALL else "tft_norain"

VAL_METRICS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_val_metrics_hourly"
WEIGHTS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_weights_hourly"
PREDICTIONS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_predictions_hourly"
METRICS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_metrics_hourly"

WRITE_DISPOSITION = "WRITE_TRUNCATE"      # atau "WRITE_APPEND"


def load_df(df, table_id, schema):
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=WRITE_DISPOSITION,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # tunggu selesai
    tbl = client.get_table(table_id)
    print(f"  {table_id}: {tbl.num_rows:,} rows")


print("Val metrics ->", VAL_METRICS_TABLE)
print("Weights     ->", WEIGHTS_TABLE)
print("Predictions ->", PREDICTIONS_TABLE)
print("Metrics     ->", METRICS_TABLE)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOOKBACK   = 72      # jam
HORIZONS   = [1, 3, 12]  # jam ke depan
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3
PATIENCE   = 7       # early stopping

# Hyperparameter TFT (selaras dgn LSTM/GRU)
TFT_HIDDEN      = 64
TFT_HEADS       = 4
TFT_DROPOUT     = 0.2
TFT_LSTM_LAYERS = 1

print(f"Device     : {DEVICE}")
print(f"Lookback   : {LOOKBACK} jam")
print(f"TFT hidden={TFT_HIDDEN} | heads={TFT_HEADS} | "
      f"dropout={TFT_DROPOUT} | lstm_layers={TFT_LSTM_LAYERS}")
print(f"Models     : {len(station_cols)} stasiun \u00d7 {len(HORIZONS)} horizon = {len(station_cols)*len(HORIZONS)}")

In [ ]:
class TMADataset(Dataset):
    def __init__(self, df, target_station, feature_cols, lookback, horizon):
        self.X, self.y = [], []
        data   = df[feature_cols].values.astype(np.float32)
        target = df[target_station].values.astype(np.float32)

        for i in range(lookback, len(data) - horizon + 1):
            self.X.append(data[i - lookback:i])          # (lookback, n_features)
            self.y.append(target[i + horizon - 1])        # scalar t+h

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## TFT (Temporal Fusion Transformer)

Implementasi TFT mengikuti arsitektur Lim et al. (2021), memakai `TMADataset`,
`train_epoch`/`eval_epoch`/`get_predictions`, `compute_metrics`, serta
konvensi per-stasiun × per-horizon yang sama seperti LSTM/GRU.

Komponen utama:

- **Variable Selection Network (VSN)** — memberi bobot per-fitur (TMA antar
  stasiun + kalender + hujan DAS target) di setiap timestep, sehingga model dapat menekan
  fitur yang tidak relevan alih-alih memperlakukan semua input setara seperti
  LSTM/GRU biasa. Ini juga menjadi sumber *interpretability* (importance fitur).
- **Gated Residual Network (GRN)** — blok non-linear dengan gating (GLU) +
  koneksi residual + LayerNorm; memungkinkan informasi melewati transformasi
  hanya bila diperlukan, menstabilkan pelatihan pada sinyal TMA yang *spiky*.
- **LSTM encoder** — memproses seluruh jendela *lookback* (72 jam) untuk
  memberi ringkasan konteks lokal (local processing) sebelum atensi.
- **Interpretable Multi-Head Attention** — menangkap dependensi jangka panjang
  antar timestep di dalam lookback.
- **Static enrichment** — pada setup ini tidak ada kovariat statis eksternal,
  sehingga enrichment memakai konteks agregat dari encoder (tetap disertakan
  agar arsitektur lengkap dan mudah diperluas).

Karena tugasnya adalah *direct single-step forecasting* (prediksi skalar
`t+h`), head TFT mereduksi keluaran atensi pada posisi terakhir menjadi satu
nilai — identik dengan cara LSTM/GRU mengambil `out[:, -1, :]`.

In [ ]:
# --- TFT building blocks -----------------------------------------------------
import torch.nn.functional as F


class GatedLinearUnit(nn.Module):
    """GLU: mengatur seberapa besar informasi diteruskan (gating)."""
    def __init__(self, input_size, hidden_size=None, dropout=0.1):
        super().__init__()
        hidden_size = hidden_size or input_size
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(input_size, hidden_size * 2)
        self.hidden_size = hidden_size

    def forward(self, x):
        x = self.dropout(x)
        x = self.fc(x)
        return F.glu(x, dim=-1)   # split terakhir jadi (value, gate)


class GatedResidualNetwork(nn.Module):
    """
    GRN(x) = LayerNorm(x + GLU(ELU(W1 x + b1) -> W2))
    Blok inti TFT: transformasi non-linear yang bisa "dilewati" via gating.
    """
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1,
                 context_size=None):
        super().__init__()
        output_size = output_size or input_size
        self.input_size = input_size
        self.output_size = output_size

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.context = (nn.Linear(context_size, hidden_size, bias=False)
                        if context_size is not None else None)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.glu = GatedLinearUnit(hidden_size, output_size, dropout)
        self.layer_norm = nn.LayerNorm(output_size)

        # proyeksi skip bila dimensi input != output
        self.skip = (nn.Linear(input_size, output_size)
                     if input_size != output_size else None)

    def forward(self, x, context=None):
        residual = x if self.skip is None else self.skip(x)
        h = self.fc1(x)
        if self.context is not None and context is not None:
            h = h + self.context(context)
        h = F.elu(h)
        h = self.fc2(h)
        h = self.glu(h)
        return self.layer_norm(h + residual)


class VariableSelectionNetwork(nn.Module):
    """
    Memberi bobot softmax per-fitur di tiap timestep, lalu menggabungkan
    representasi tiap fitur (masing-masing lewat GRN) dengan bobot tsb.
    Input : (batch, time, n_features)
    Output: (batch, time, hidden_size), plus bobot (batch, time, n_features)
    """
    def __init__(self, n_features, hidden_size, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size

        # GRN untuk menghitung bobot seleksi (dari seluruh fitur flatten)
        self.flattened_grn = GatedResidualNetwork(
            n_features, hidden_size, n_features, dropout
        )
        # GRN per-fitur (skalar -> hidden)
        self.per_feature_grn = nn.ModuleList([
            GatedResidualNetwork(1, hidden_size, hidden_size, dropout)
            for _ in range(n_features)
        ])

    def forward(self, x):
        # x: (B, T, F)
        weights = self.flattened_grn(x)                 # (B, T, F)
        weights = torch.softmax(weights, dim=-1).unsqueeze(-1)  # (B, T, F, 1)

        transformed = torch.stack([
            self.per_feature_grn[i](x[..., i:i + 1])    # (B, T, hidden)
            for i in range(self.n_features)
        ], dim=-2)                                       # (B, T, F, hidden)

        combined = (transformed * weights).sum(dim=-2)   # (B, T, hidden)
        return combined, weights.squeeze(-1)


class InterpretableMultiHeadAttention(nn.Module):
    """
    Multi-head attention dgn value bersama antar head (versi interpretable TFT).
    """
    def __init__(self, hidden_size, n_heads, dropout=0.1):
        super().__init__()
        assert hidden_size % n_heads == 0
        self.n_heads = n_heads
        self.d_head = hidden_size // n_heads

        self.q = nn.Linear(hidden_size, hidden_size)
        self.k = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, self.d_head)     # value dibagi antar head
        self.out = nn.Linear(self.d_head, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v(x).unsqueeze(1)                        # (B,1,T,d_head)

        scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        ctx = attn @ v                                    # (B,H,T,d_head)
        ctx = ctx.mean(dim=1)                             # rata-rata antar head
        return self.out(ctx), attn


class TFTForecaster(nn.Module):
    """
    Temporal Fusion Transformer untuk direct single-step forecasting.
    Signature forward identik dgn LSTM/GRUForecaster: x (B, LOOKBACK, F) -> (B,)
    """
    def __init__(self, n_features, hidden_size=64, n_heads=4,
                 dropout=0.2, lstm_layers=1):
        super().__init__()
        self.vsn = VariableSelectionNetwork(n_features, hidden_size, dropout)

        self.lstm_encoder = nn.LSTM(
            input_size=hidden_size, hidden_size=hidden_size,
            num_layers=lstm_layers, batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        # gate + norm setelah encoder (post-LSTM gating TFT)
        self.post_lstm_glu = GatedLinearUnit(hidden_size, hidden_size, dropout)
        self.post_lstm_norm = nn.LayerNorm(hidden_size)

        # static enrichment memakai konteks agregat encoder
        self.enrichment = GatedResidualNetwork(
            hidden_size, hidden_size, hidden_size, dropout,
            context_size=hidden_size,
        )
        self.attention = InterpretableMultiHeadAttention(
            hidden_size, n_heads, dropout
        )
        self.post_attn_glu = GatedLinearUnit(hidden_size, hidden_size, dropout)
        self.post_attn_norm = nn.LayerNorm(hidden_size)

        self.position_wise = GatedResidualNetwork(
            hidden_size, hidden_size, hidden_size, dropout
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x, return_weights=False):
        # 1) variable selection
        selected, var_weights = self.vsn(x)              # (B,T,H)

        # 2) local processing (LSTM encoder)
        lstm_out, (h_n, _) = self.lstm_encoder(selected)
        lstm_out = self.post_lstm_norm(
            self.post_lstm_glu(lstm_out) + selected      # residual skip
        )

        # 3) static enrichment (konteks = hidden terakhir encoder)
        context = h_n[-1].unsqueeze(1).expand(-1, lstm_out.size(1), -1)
        enriched = self.enrichment(lstm_out, context)    # (B,T,H)

        # 4) temporal self-attention
        attn_out, attn_weights = self.attention(enriched)
        attn_out = self.post_attn_norm(
            self.post_attn_glu(attn_out) + enriched
        )

        # 5) position-wise feed-forward + head pada timestep terakhir
        out = self.position_wise(attn_out)
        pred = self.head(out[:, -1, :]).squeeze(-1)       # (B,)

        if return_weights:
            return pred, var_weights, attn_weights
        return pred

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)
            total_loss += criterion(pred, y).item() * len(y)
    return total_loss / len(loader.dataset)


def get_predictions(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).cpu().numpy())
            trues.append(y.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def compute_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    # sMAPE \u2014 symmetric, aman untuk nilai mendekati 0
    smape = np.mean(
        2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # NSE \u2014 Nash-Sutcliffe Efficiency
    nse = 1 - (np.sum((y_true - y_pred) ** 2) /
               np.sum((y_true - np.mean(y_true)) ** 2 + 1e-8))

    return {"RMSE": rmse, "MAE": mae, "sMAPE": smape, "NSE": nse}

In [ ]:
tft_results = {}
tft_models  = {}

criterion = nn.MSELoss()

for target in station_cols:
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    tft_models[target]  = {}
    tft_results[target] = {}

    for h in HORIZONS:
        train_ds = TMADataset(df_train, target, features_for(target), LOOKBACK, h)
        val_ds   = TMADataset(df_val,   target, features_for(target), LOOKBACK, h)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = TFTForecaster(
            n_features=len(features_for(target)),
            hidden_size=TFT_HIDDEN,
            n_heads=TFT_HEADS,
            dropout=TFT_DROPOUT,
            lstm_layers=TFT_LSTM_LAYERS,
        ).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_val_loss = float("inf")
        best_state    = None
        patience_ctr  = 0

        for epoch in range(EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_loss   = eval_epoch(model, val_loader, criterion)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr  = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"  h+{h:>2}h | early stop epoch {epoch+1:>3} | "
                          f"val_loss={best_val_loss:.6f}")
                    break

        model.load_state_dict(best_state)
        tft_models[target][h] = model

        y_pred, y_true = get_predictions(model, val_loader)
        metrics = compute_metrics(y_true, y_pred)
        tft_results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}h \u2192 RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")

In [ ]:
weights_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
    bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]
val_metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]


def serialize_torch_state(state_dict):
    """Serialize state_dict PyTorch ke bytes (in-memory, tanpa file sementara)."""
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    return buf.getvalue()


saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

weight_rows = []
val_metric_rows = []
for station, horizons in tft_models.items():
    for h, model in horizons.items():
        blob = serialize_torch_state(model.state_dict())
        weight_rows.append({
            "model": MODEL_NAME, "station": station, "horizon": int(h),
            "weights": blob, "n_bytes": len(blob), "saved_at": saved_at,
        })
    for hkey, metrics in tft_results[station].items():
        h = int(hkey.replace("h+", ""))
        for metric_name, value in metrics.items():
            val_metric_rows.append({
                "model": MODEL_NAME, "station": station, "horizon": h,
                "metric": metric_name, "value": float(value),
            })

weights_df = pd.DataFrame(weight_rows)
val_metrics_df = pd.DataFrame(val_metric_rows)
total_mb = weights_df["n_bytes"].sum() / 1e6
print(f"Menyiapkan {len(weights_df)} baris bobot ({total_mb:.2f} MB total).")

load_df(weights_df, WEIGHTS_TABLE, weights_schema)
load_df(val_metrics_df, VAL_METRICS_TABLE, val_metric_schema)
print(f"\n\u2714 Bobot + metrik validasi '{MODEL_NAME}' tersimpan di BigQuery.")

## Test Set Evaluation

In [ ]:
def _load_weights_from_bq(station, h):
    query = f"""
        SELECT weights
        FROM `{WEIGHTS_TABLE}`
        WHERE station = @station AND horizon = @horizon
        ORDER BY saved_at DESC
        LIMIT 1
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("station", "STRING", station),
        bigquery.ScalarQueryParameter("horizon", "INT64", h),
    ])
    row = client.query(query, job_config=job_config).to_dataframe()
    if row.empty:
        raise RuntimeError(f"Bobot utk ({station}, h+{h}) tidak ditemukan di {WEIGHTS_TABLE}.")
    return row.iloc[0]["weights"]


def predict_test_tft(target, h):
    test_ds = TMADataset(df_test, target, features_for(target), LOOKBACK, h)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    if target in tft_models and h in tft_models[target]:
        model = tft_models[target][h]
    else:
        # rebuild + load bobot dari BigQuery (mis. notebook baru dibuka ulang)
        model = TFTForecaster(
            n_features=len(features_for(target)), hidden_size=TFT_HIDDEN,
            n_heads=TFT_HEADS, dropout=TFT_DROPOUT,
            lstm_layers=TFT_LSTM_LAYERS,
        ).to(DEVICE)
        blob = _load_weights_from_bq(target, h)
        model.load_state_dict(torch.load(io.BytesIO(blob), map_location=DEVICE))

    idx = df_test.index[LOOKBACK + h - 1 : len(df_test)]
    y_pred, y_true = get_predictions(model, test_loader)
    return idx[:len(y_pred)], y_true, y_pred


test_results = {}
test_predictions = []   # baris panjang utk BigQuery

for target in station_cols:
    test_results[target] = {}
    for h in HORIZONS:
        idx, y_true, y_pred = predict_test_tft(target, h)
        metrics = compute_metrics(y_true, y_pred)
        test_results[target][f"h+{h}"] = metrics

        for ts, yt, yp in zip(idx, y_true, y_pred):
            test_predictions.append({
                "model": MODEL_NAME,
                "station": target,
                "horizon": int(h),
                "target_datetime": pd.Timestamp(ts),
                "y_true": float(yt),
                "y_pred": float(yp),
            })

    row = " | ".join(
        f"h+{h}: NSE={test_results[target][f'h+{h}']['NSE']:.3f}"
        for h in HORIZONS
    )
    print(f"{target:<20} {row}")

test_pred_df = pd.DataFrame(test_predictions)
print(f"\nTest predictions: {test_pred_df.shape[0]:,} rows.")
test_pred_df.head()

In [ ]:
metric_rows = []
for station, horizons in test_results.items():
    for hkey, metrics in horizons.items():
        for metric_name, value in metrics.items():
            metric_rows.append({
                "model": MODEL_NAME, "station": station,
                "horizon": int(hkey.replace("h+", "")),
                "metric": metric_name, "value": float(value),
            })
test_metrics_df = pd.DataFrame(metric_rows)
print(f"Test metrics rows: {len(test_metrics_df)}")

## Write Results to BigQuery

Prediksi dan metrik test-set model ini ditulis ke BigQuery. Autentikasi
mengikuti mekanisme standar Google Cloud (Application Default Credentials,
mis. kredensial bawaan Colab). Mode tulis default `WRITE_TRUNCATE` (ganti
seluruh isi tabel); ubah `WRITE_DISPOSITION` di atas ke `WRITE_APPEND` bila
ingin menumpuk hasil beberapa run.

In [ ]:
pred_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("target_datetime", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("y_true", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("y_pred", "FLOAT", mode="NULLABLE"),
]
metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]

# target_datetime -> pastikan tz-naive UTC agar konsisten di BQ
_pred = test_pred_df.copy()
_pred["target_datetime"] = pd.to_datetime(_pred["target_datetime"])

print("Uploading to BigQuery ...")
load_df(_pred, PREDICTIONS_TABLE, pred_schema)
load_df(test_metrics_df, METRICS_TABLE, metric_schema)
print("\n\u2714 BigQuery write selesai.")

## Eksperimen: MIMO Multi-Horizon + Pinball (Quantile) Loss

Model TFT utama di atas memakai strategi **direct**: satu model per (stasiun,
horizon) yang dilatih dengan MSE, total 24 model. TFT versi asli (Lim et al.,
2021) justru dirancang multi-horizon dan probabilistik: satu model memprediksi
semua horizon sekaligus, dengan output kuantil yang dilatih memakai *quantile
(pinball) loss*. Section ini menguji dua varian itu:

| Strategi | Jumlah model | Output per sampel | Loss |
|---|---|---|---|
| `direct` | 24 (stasiun × horizon) | `y(t+h)` | MSE |
| `mimo` | 8 (per stasiun) | `[y(t+1), y(t+3), y(t+12)]` | MSE, rata-rata 3 horizon |
| `mimo_q` | 8 (per stasiun) | kuantil 0.1 / 0.5 / 0.9 tiap horizon | pinball, rata-rata 3 horizon × 3 kuantil |

- `direct` vs `mimo` mengukur efek strategi multi-horizon. Desainnya sama persis
  dengan eksperimen MIMO di notebook LSTM/GRU.
- `mimo` vs `mimo_q` mengukur efek loss (MSE vs pinball) dengan strategi yang sama.
- Varian direct + pinball tidak dilatih (perlu 24 model tambahan), jadi efek
  pinball hanya diuji pada setting MIMO.

**Pinball loss** untuk kuantil `q`: `L_q(y, ŷ) = max(q·(y − ŷ), (q − 1)·(y − ŷ))`.
Prediksi titik `mimo_q` adalah median (`q = 0.5`), sehingga NSE/RMSE-nya bisa
dibandingkan langsung dengan dua strategi lain. Perlu dicatat: median
meminimalkan MAE, bukan MSE, jadi metrik berbasis kuadrat error (NSE, RMSE)
cenderung sedikit menguntungkan model yang dilatih dengan MSE.

Interval `[q0.1, q0.9]` adalah *prediction interval* (PI) 80%, dievaluasi dengan:

- **PICP** (coverage, %): porsi `y_true` di dalam interval. Idealnya ≈ 80%.
- **MPIW**: rata-rata lebar interval (skala 0–1). Makin sempit makin baik, asal
  coverage terpenuhi.
- **PICP_high** (%): coverage pada 5% `y_true` tertinggi di test set, yaitu
  kondisi TMA tinggi yang paling penting untuk peringatan dini banjir.
- **pinball**: rata-rata pinball loss atas ketiga kuantil (makin kecil makin baik).
- **crossing** (%): porsi sampel yang kuantilnya bersilangan (mis. q0.1 > q0.5)
  sebelum diurutkan. Kuantil selalu diurutkan setelah prediksi.

Supaya perbandingannya adil, semua hal lain dibuat sama: fitur input per
stasiun, lookback, arsitektur TFT (hidden 64, 4 head, dropout 0.2, 1 layer
LSTM), optimizer, batch size, epoch maksimum, dan early stopping (memakai val
loss gabungan). MIMO butuh target hingga `t + max(HORIZONS)`, jadi window
terakhir di tiap split sedikit lebih sedikit. Evaluasi test di bawah
membandingkan ketiga strategi pada **window input yang persis sama**.

Model TFT direct diambil dari memori bila ada; bila tidak (mis. notebook baru
dibuka), bobotnya dimuat dari BigQuery lewat `predict_test_tft`. Jadi section ini
bisa dijalankan tanpa melatih ulang model direct: jalankan cell setup di bagian
atas (data, konfigurasi tabel, hyperparameter, dataset, building block TFT,
helper), lalu cell pertama *Test Set Evaluation* (yang mendefinisikan
`predict_test_tft`), lalu section ini. Jangan jalankan cell upload bobot/hasil
direct bila tidak ingin menimpa tabel `tft_*`.

Hasil section ini **tidak dipakai pipeline produksi** (`src/prediction` memuat
TFT dengan head satu output). Secara default tidak ada yang ditulis ke BigQuery
(`SAVE_MIMO_TO_BQ = False`). Bila diaktifkan, hasilnya masuk ke tabel terpisah
`tft_mimo_*_hourly` dan `tft_mimo_q_*_hourly`, yang tidak dibaca oleh notebook
model selection maupun `src/prediction`.

Catatan: model direct dilatih tanpa seed, dan setiap model hanya dilatih satu
kali. Selisih NSE yang kecil bisa saja masih di dalam variasi antar-run; lihat
bagian multi-seed di bawah untuk perbandingan yang memperhitungkan variasi itu.

In [ ]:
import time

MIMO_MODEL_NAME   = f"{MODEL_NAME}_mimo"     # "tft_mimo"   / "tft_norain_mimo"
MIMO_Q_MODEL_NAME = f"{MODEL_NAME}_mimo_q"   # "tft_mimo_q" / "tft_norain_mimo_q"
QUANTILES         = [0.1, 0.5, 0.9]
assert 0.5 in QUANTILES and QUANTILES == sorted(QUANTILES)
Q_MEDIAN          = QUANTILES.index(0.5)
NOMINAL_PI        = round((QUANTILES[-1] - QUANTILES[0]) * 100)   # 80 (%)
H_MAX             = max(HORIZONS)
MIMO_SEED         = 42
SAVE_MIMO_TO_BQ   = False   # True = simpan ke tabel {MIMO_MODEL_NAME}_* dan {MIMO_Q_MODEL_NAME}_*

STRATEGIES = ["direct", "mimo", "mimo_q"]
PAIRS      = [("mimo", "direct"), ("mimo_q", "mimo"), ("mimo_q", "direct")]   # (a, b): delta = a - b
TFT_KW     = dict(hidden_size=TFT_HIDDEN, n_heads=TFT_HEADS, dropout=TFT_DROPOUT,
                  lstm_layers=TFT_LSTM_LAYERS)
S_COLORS   = {"direct": "#7f7f7f", "mimo": "#d62728", "mimo_q": "#1f77b4"}
S_LABELS   = {"direct": "Direct, MSE (24 model)", "mimo": "MIMO, MSE (8 model)",
              "mimo_q": "MIMO, pinball q0.5 (8 model)"}

# Bila model direct belum dilatih di sesi ini, predict_test_tft memuatnya dari BigQuery.
if "tft_models" not in globals():
    tft_models = {}


class TMADatasetMIMO(Dataset):
    # Satu window input -> vektor target [y(t+h) for h in HORIZONS].
    # Window ke-k memakai indeks i = lookback + k, sama seperti TMADataset,
    # sehingga sampel ke-k di kedua dataset berasal dari window input yang sama.
    def __init__(self, df, target_station, feature_cols, lookback, horizons):
        data   = df[feature_cols].values.astype(np.float32)
        target = df[target_station].values.astype(np.float32)
        h_max  = max(horizons)

        X, y = [], []
        for i in range(lookback, len(data) - h_max + 1):
            X.append(data[i - lookback:i])                   # (lookback, n_features)
            y.append([target[i + h - 1] for h in horizons])  # (n_horizons,)

        self.X = torch.tensor(np.array(X), dtype=torch.float32)
        self.y = torch.tensor(np.array(y), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class TFTForecasterMulti(TFTForecaster):
    # Identik dengan TFTForecaster, kecuali head: hidden -> n_horizons (x n_quantiles).
    # Output (B, n_horizons) bila quantiles=None, (B, n_horizons, n_quantiles) bila ada.
    def __init__(self, n_features, n_horizons, quantiles=None, **kwargs):
        super().__init__(n_features, **kwargs)
        self.n_horizons  = n_horizons
        self.n_quantiles = len(quantiles) if quantiles else 0
        self.head = nn.Linear(self.head.in_features, n_horizons * max(self.n_quantiles, 1))

    def forward(self, x, return_weights=False):
        out  = super().forward(x, return_weights=return_weights)
        pred = out[0] if return_weights else out
        if self.n_quantiles:
            pred = pred.view(-1, self.n_horizons, self.n_quantiles)
        return (pred, *out[1:]) if return_weights else pred


def make_tft(target, multi=False, quantiles=None):
    n_feat = len(features_for(target))
    if not multi:
        return TFTForecaster(n_features=n_feat, **TFT_KW).to(DEVICE)
    return TFTForecasterMulti(n_feat, n_horizons=len(HORIZONS), quantiles=quantiles, **TFT_KW).to(DEVICE)


class PinballLoss(nn.Module):
    # Quantile (pinball) loss, dirata-rata atas sampel, horizon, dan kuantil.
    # pred: (B, n_horizons, n_quantiles), y: (B, n_horizons).
    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = list(quantiles)

    def forward(self, pred, y):
        q = torch.tensor(self.quantiles, dtype=pred.dtype, device=pred.device)
        e = y.unsqueeze(-1) - pred
        return torch.maximum(q * e, (q - 1) * e).mean()


mse_loss     = nn.MSELoss()   # MSELoss pada (B, n_horizons) = rata-rata error atas semua horizon
pinball_loss = PinballLoss(QUANTILES)


def fit(model, train_loader, val_loader, criterion):
    # Training + early stopping, sama dengan loop training direct di atas.
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_val_loss, best_state, patience_ctr = float("inf"), None, 0
    t0 = time.perf_counter()
    for epoch in range(EPOCHS):
        train_epoch(model, train_loader, optimizer, criterion)
        val_loss = eval_epoch(model, val_loader, criterion)
        if val_loss < best_val_loss:
            best_val_loss, patience_ctr = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, {"epochs": epoch + 1, "seconds": time.perf_counter() - t0, "val_loss": best_val_loss}


def get_quantile_predictions(model, loader):
    # Kuantil diurutkan per sampel (q0.1 <= q0.5 <= q0.9). crossing = porsi sampel
    # per horizon yang kuantilnya bersilangan sebelum diurutkan, shape (n_horizons,).
    q_pred, y_true = get_predictions(model, loader)               # (N, H, Q), (N, H)
    crossing = (np.diff(q_pred, axis=-1) < 0).any(axis=-1).mean(axis=0)
    return np.sort(q_pred, axis=-1), y_true, crossing


def interval_metrics(y_true, q_pred, quantiles, high_frac=0.05):
    # y_true (N,), q_pred (N, Q) sudah diurutkan. Interval = [kuantil terendah, tertinggi].
    mask = ~np.isnan(y_true) & ~np.isnan(q_pred).any(axis=1)
    y, qp = y_true[mask], q_pred[mask]
    lo, hi = qp[:, 0], qp[:, -1]
    inside = (y >= lo) & (y <= hi)
    high = y >= np.quantile(y, 1 - high_frac)
    e = y[:, None] - qp
    q = np.asarray(quantiles)[None, :]
    return {
        "pinball":   float(np.mean(np.maximum(q * e, (q - 1) * e))),
        "PICP":      float(inside.mean() * 100),
        "PICP_high": float(inside[high].mean() * 100),
        "MPIW":      float((hi - lo).mean()),
    }


def n_params(model):
    return sum(p.numel() for p in model.parameters())


print(f"Eksperimen : {MIMO_MODEL_NAME} (MSE) dan {MIMO_Q_MODEL_NAME} (pinball, kuantil {QUANTILES})")
print(f"Models     : {len(station_cols)} stasiun x 1 model (output {HORIZONS}) = {len(station_cols)} per strategi")

In [ ]:
# --- Training MIMO (MSE) dan MIMO-quantile (pinball): satu model per stasiun --
mimo_models,   mimo_val_results,   mimo_train_info   = {}, {}, {}
mimo_q_models, mimo_q_val_results, mimo_q_train_info = {}, {}, {}

for target in station_cols:
    feats = features_for(target)
    train_loader = DataLoader(TMADatasetMIMO(df_train, target, feats, LOOKBACK, HORIZONS),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TMADatasetMIMO(df_val, target, feats, LOOKBACK, HORIZONS),
                              batch_size=BATCH_SIZE, shuffle=False)

    # MIMO, MSE
    torch.manual_seed(MIMO_SEED)
    model, info = fit(make_tft(target, multi=True), train_loader, val_loader, mse_loss)
    mimo_models[target], mimo_train_info[target] = model, info
    yp, yt = get_predictions(model, val_loader)                    # (N, n_horizons)
    mimo_val_results[target] = {
        f"h+{h}": compute_metrics(yt[:, j], yp[:, j]) for j, h in enumerate(HORIZONS)
    }

    # MIMO, pinball (prediksi titik = median)
    torch.manual_seed(MIMO_SEED)
    model, info_q = fit(make_tft(target, multi=True, quantiles=QUANTILES),
                        train_loader, val_loader, pinball_loss)
    mimo_q_models[target], mimo_q_train_info[target] = model, info_q
    qp, yt, _ = get_quantile_predictions(model, val_loader)        # (N, n_horizons, n_quantiles)
    mimo_q_val_results[target] = {
        f"h+{h}": {**compute_metrics(yt[:, j], qp[:, j, Q_MEDIAN]),
                   **interval_metrics(yt[:, j], qp[:, j], QUANTILES)}
        for j, h in enumerate(HORIZONS)
    }

    row = " | ".join(
        f"h+{h}: NSE {mimo_val_results[target][f'h+{h}']['NSE']:.3f}/"
        f"{mimo_q_val_results[target][f'h+{h}']['NSE']:.3f} "
        f"PICP {mimo_q_val_results[target][f'h+{h}']['PICP']:.0f}%"
        for h in HORIZONS
    )
    print(f"{target:<20} {row}  ({info['epochs']}+{info_q['epochs']} epoch, "
          f"{info['seconds'] + info_q['seconds']:.0f} s)")

print("\nFormat: NSE val mimo/mimo_q, PICP val mimo_q (target "
      f"{NOMINAL_PI}%); epoch mimo+mimo_q.")

In [ ]:
# --- Test set: direct vs MIMO vs MIMO-quantile pada window input yang sama ----
comparison_rows         = []   # (strategy, station, horizon, metric, value), metrik titik
interval_rows           = []   # (station, horizon, metric, value), metrik interval mimo_q
mimo_test_predictions   = []   # format sama dengan test_predictions direct
mimo_q_test_predictions = []   # + kolom y_lower / y_upper (batas PI)

for target in station_cols:
    feats       = features_for(target)
    test_loader = DataLoader(TMADatasetMIMO(df_test, target, feats, LOOKBACK, HORIZONS),
                             batch_size=BATCH_SIZE, shuffle=False)
    n = len(test_loader.dataset)
    yp_m, yt_m           = get_predictions(mimo_models[target], test_loader)
    qp, yt_q, crossing   = get_quantile_predictions(mimo_q_models[target], test_loader)

    for j, h in enumerate(HORIZONS):
        # direct: n window pertama = window yang sama dengan dataset MIMO
        idx, yt_d, yp_d = predict_test_tft(target, h)
        idx, yt_d, yp_d = idx[:n], yt_d[:n], yp_d[:n]
        assert np.allclose(yt_d, yt_m[:, j], equal_nan=True), f"target direct vs MIMO tidak sejajar ({target}, h+{h})"
        assert np.allclose(yt_d, yt_q[:, j], equal_nan=True), f"target direct vs MIMO-q tidak sejajar ({target}, h+{h})"

        for strategy, yp in (("direct", yp_d), ("mimo", yp_m[:, j]), ("mimo_q", qp[:, j, Q_MEDIAN])):
            for metric_name, value in compute_metrics(yt_d, yp).items():
                comparison_rows.append({"strategy": strategy, "station": target, "horizon": int(h),
                                        "metric": metric_name, "value": float(value)})

        iv = {**interval_metrics(yt_d, qp[:, j], QUANTILES), "crossing": float(crossing[j] * 100)}
        for metric_name, value in iv.items():
            interval_rows.append({"station": target, "horizon": int(h),
                                  "metric": metric_name, "value": float(value)})

        for ts, yt, ym, ql, qm, qh in zip(idx, yt_d, yp_m[:, j], qp[:, j, 0], qp[:, j, Q_MEDIAN], qp[:, j, -1]):
            base = {"station": target, "horizon": int(h), "target_datetime": pd.Timestamp(ts),
                    "y_true": float(yt)}
            mimo_test_predictions.append({"model": MIMO_MODEL_NAME, **base, "y_pred": float(ym)})
            mimo_q_test_predictions.append({"model": MIMO_Q_MODEL_NAME, **base, "y_pred": float(qm),
                                            "y_lower": float(ql), "y_upper": float(qh)})

comparison_df       = pd.DataFrame(comparison_rows)
interval_df         = pd.DataFrame(interval_rows)
mimo_test_pred_df   = pd.DataFrame(mimo_test_predictions)
mimo_q_test_pred_df = pd.DataFrame(mimo_q_test_predictions)
print(f"Perbandingan: {comparison_df.shape[0]} baris metrik titik, "
      f"{len(station_cols)} stasiun x {len(HORIZONS)} horizon x {len(STRATEGIES)} strategi; "
      f"{interval_df.shape[0]} baris metrik interval.")

In [ ]:
# --- Ringkasan perbandingan ---------------------------------------------------
from scipy.stats import wilcoxon

nse = (comparison_df[comparison_df.metric == "NSE"]
       .pivot_table(index=["station", "horizon"], columns="strategy", values="value"))[STRATEGIES]
for a, b in PAIRS:
    nse[f"{a}-{b}"] = nse[a] - nse[b]   # > 0 = strategi a lebih baik

print("NSE test per stasiun x horizon (kolom a-b = selisih NSE, > 0 = a lebih baik):")
display(nse.round(4))

# Mean & median tiap metrik per horizon. NSE: lebih tinggi lebih baik; lainnya: lebih rendah.
summary = (comparison_df.groupby(["metric", "horizon", "strategy"])["value"]
           .agg(["mean", "median"]).unstack("strategy"))
print("\nRingkasan metrik titik test per horizon:")
display(summary.round(4))

# Wilcoxon signed-rank berpasangan per horizon (n = 8 stasiun). Dengan n = 8,
# p terkecil yang mungkin (two-sided) adalah 2/256 = 0.0078, yaitu bila
# kedelapan stasiun selisihnya searah.
for a, b in PAIRS:
    print(f"\n{a} vs {b}:")
    for h in HORIZONS:
        d = nse.xs(h, level="horizon")[f"{a}-{b}"]
        print(f"  h+{h:<2}: {a} menang {int((d > 0).sum())}/{len(d)} | "
              f"median delta NSE = {d.median():+.4f} | Wilcoxon p={wilcoxon(d).pvalue:.4f}")

# Kualitas interval prediksi mimo_q.
iv_tbl = (interval_df.pivot_table(index=["station", "horizon"], columns="metric", values="value")
          [["PICP", "PICP_high", "MPIW", "pinball", "crossing"]])
print(f"\nPrediction interval {NOMINAL_PI}% (mimo_q) per stasiun x horizon "
      f"(PICP, PICP_high, crossing dalam %):")
display(iv_tbl.round(4))
print("\nRata-rata per horizon:")
display(iv_tbl.groupby(level="horizon").mean().round(4))

# Efisiensi: jumlah model, parameter, dan waktu training.
direct_params = sum(n_params(TFTForecaster(n_features=len(features_for(s)), **TFT_KW))
                    for s in station_cols) * len(HORIZONS)
mimo_params   = sum(n_params(m) for m in mimo_models.values())
mimo_q_params = sum(n_params(m) for m in mimo_q_models.values())
print(f"\nJumlah model    : direct {len(station_cols) * len(HORIZONS)} | mimo {len(mimo_models)} | "
      f"mimo_q {len(mimo_q_models)}")
print(f"Total parameter : direct {direct_params:,} | mimo {mimo_params:,} "
      f"({mimo_params / direct_params:.1%}) | mimo_q {mimo_q_params:,} ({mimo_q_params / direct_params:.1%})")
print(f"Waktu training  : mimo {sum(i['seconds'] for i in mimo_train_info.values()) / 60:.1f} menit | "
      f"mimo_q {sum(i['seconds'] for i in mimo_q_train_info.values()) / 60:.1f} menit "
      f"(waktu training direct tidak dicatat di notebook ini)")

In [ ]:
# --- Visualisasi ----------------------------------------------------------------
import matplotlib.pyplot as plt

H_MARKERS = dict(zip(HORIZONS, ["o", "s", "^"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 1) NSE varian MIMO vs direct per (stasiun, horizon); di atas garis = lebih baik dari direct
for s in ("mimo", "mimo_q"):
    for h in HORIZONS:
        d = nse.xs(h, level="horizon")
        ax1.scatter(d["direct"], d[s], color=S_COLORS[s], marker=H_MARKERS[h], s=42, alpha=0.8,
                    label=f"{s}, h+{h}")
lo = nse[STRATEGIES].min().min()
lo = lo - 0.05 * (1 - lo)
ax1.plot([lo, 1], [lo, 1], color="k", lw=0.8, ls="--")
ax1.set_xlim(lo, 1)
ax1.set_ylim(lo, 1)
ax1.set_xlabel("NSE direct")
ax1.set_ylabel("NSE varian MIMO")
ax1.set_title("NSE test per stasiun x horizon")
ax1.legend(fontsize=7, ncol=2)

# 2) Median NSE per horizon (median karena stasiun dengan NSE rendah menarik rata-rata)
med = nse.groupby(level="horizon")[STRATEGIES].median()
x = np.arange(len(HORIZONS))
w = 0.27
for k, s in enumerate(STRATEGIES):
    off = (k - (len(STRATEGIES) - 1) / 2) * w
    ax2.bar(x + off, med[s], w, label=S_LABELS[s], color=S_COLORS[s])
    for i, h in enumerate(HORIZONS):
        ax2.text(i + off, med.loc[h, s], f"{med.loc[h, s]:.3f}", ha="center", va="bottom", fontsize=7)
ax2.set_xticks(x, [f"h+{h}" for h in HORIZONS])
ax2.set_ylabel("Median NSE (test)")
ax2.set_title("Median NSE per horizon")
ax2.legend(loc="lower left", fontsize=8)

fig.suptitle(f"{MODEL_NAME.upper()}: direct vs MIMO vs MIMO + pinball loss")
fig.tight_layout()
plt.show()

# 3) Prediction interval di sekitar TMA puncak tiap stasiun (test set)
PI_HORIZON = HORIZONS[1]
PI_WINDOW  = 48   # jam sebelum & sesudah puncak

q_df = mimo_q_test_pred_df[mimo_q_test_pred_df.horizon == PI_HORIZON]
n_cols = 2
n_rows = int(np.ceil(len(station_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 2.8 * n_rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, target in zip(axes.ravel(), station_cols):
    d = q_df[q_df.station == target].set_index("target_datetime").sort_index()
    peak = d["y_true"].idxmax()
    d = d.loc[peak - pd.Timedelta(hours=PI_WINDOW): peak + pd.Timedelta(hours=PI_WINDOW)]
    ax.axis("on")
    ax.fill_between(d.index, d["y_lower"], d["y_upper"], color=S_COLORS["mimo_q"], alpha=0.25,
                    label=f"PI {NOMINAL_PI}%")
    ax.plot(d.index, d["y_pred"], color=S_COLORS["mimo_q"], lw=1.2, label="median")
    ax.plot(d.index, d["y_true"], color="k", lw=1.0, label="aktual")
    ax.set_title(f"{target}  |  puncak {peak:%Y-%m-%d %H:%M}", fontsize=9)
    ax.tick_params(axis="x", labelsize=7, rotation=30)
axes.ravel()[0].legend(fontsize=8, frameon=False)
fig.suptitle(f"{MIMO_Q_MODEL_NAME.upper()}: prediction interval {NOMINAL_PI}% di sekitar TMA puncak "
             f"(h+{PI_HORIZON}, skala 0-1)")
fig.tight_layout()
plt.show()

In [ ]:
# --- (Opsional) simpan hasil MIMO & MIMO-quantile ke BigQuery, tabel terpisah --
if SAVE_MIMO_TO_BQ:
    _metric_schema = [
        bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
    ]
    _pred_schema = [
        bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("target_datetime", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("y_true", "FLOAT", mode="NULLABLE"),
        bigquery.SchemaField("y_pred", "FLOAT", mode="NULLABLE"),
    ]
    _pred_q_schema = _pred_schema + [
        bigquery.SchemaField("y_lower", "FLOAT", mode="NULLABLE"),   # kuantil terendah
        bigquery.SchemaField("y_upper", "FLOAT", mode="NULLABLE"),   # kuantil tertinggi
    ]
    _weights_schema = [
        bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("horizon", "INTEGER", mode="NULLABLE"),   # NULL: satu model utk semua horizon
        bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
        bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
    ]
    _saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

    def _weights_rows(name, models):
        rows = []
        for s, model in models.items():
            buf = io.BytesIO()
            torch.save(model.state_dict(), buf)
            rows.append({"model": name, "station": s, "horizon": None, "weights": buf.getvalue(),
                         "n_bytes": buf.getbuffer().nbytes, "saved_at": _saved_at})
        df = pd.DataFrame(rows)
        df["horizon"] = df["horizon"].astype("Int64")
        return df

    def _val_metric_rows(name, results):
        return pd.DataFrame([
            {"model": name, "station": s, "horizon": int(hkey.replace("h+", "")),
             "metric": m, "value": float(v)}
            for s, by_h in results.items() for hkey, metrics in by_h.items() for m, v in metrics.items()
        ])

    def _test_metric_rows(name, strategy):
        df = comparison_df[comparison_df.strategy == strategy].drop(columns="strategy")
        if strategy == "mimo_q":
            df = pd.concat([df, interval_df], ignore_index=True)   # + PICP, MPIW, pinball, ...
        return df.assign(model=name)[["model", "station", "horizon", "metric", "value"]]

    for name, strategy, models, val_results, pred_df, pred_schema in (
        (MIMO_MODEL_NAME,   "mimo",   mimo_models,   mimo_val_results,   mimo_test_pred_df,   _pred_schema),
        (MIMO_Q_MODEL_NAME, "mimo_q", mimo_q_models, mimo_q_val_results, mimo_q_test_pred_df, _pred_q_schema),
    ):
        print(f"Uploading hasil {name} ke BigQuery ...")
        _pred = pred_df.copy()
        _pred["target_datetime"] = pd.to_datetime(_pred["target_datetime"])
        load_df(_weights_rows(name, models), f"{PROJECT_ID}.{DATASET}.{name}_weights_hourly", _weights_schema)
        load_df(_val_metric_rows(name, val_results),
                f"{PROJECT_ID}.{DATASET}.{name}_val_metrics_hourly", _metric_schema)
        load_df(_test_metric_rows(name, strategy), f"{PROJECT_ID}.{DATASET}.{name}_metrics_hourly", _metric_schema)
        load_df(_pred, f"{PROJECT_ID}.{DATASET}.{name}_predictions_hourly", pred_schema)
    print(f"\n✔ Hasil '{MIMO_MODEL_NAME}' dan '{MIMO_Q_MODEL_NAME}' tersimpan di BigQuery.")
else:
    print("SAVE_MIMO_TO_BQ = False: hasil MIMO tidak ditulis ke BigQuery.")

### Multi-seed: seberapa besar variasi antar-run?

Perbandingan di atas hanya memakai satu run per strategi (direct: run produksi
tanpa seed; MIMO dan MIMO-quantile: seed 42). Jadi belum jelas apakah selisih
NSE-nya nyata atau hanya akibat inisialisasi bobot dan urutan batch. Di sini
strategi di `SEED_STRATEGIES` dilatih ulang dengan beberapa seed (`SEEDS`), lalu
dievaluasi pada window test yang sama seperti di atas.

- Setiap seed melatih 8 model untuk tiap varian MIMO, ditambah 24 model direct
  bila `"direct"` ada di `SEED_STRATEGIES`. Model direct dari run ini **tidak**
  menggantikan model produksi dan tidak diunggah ke tabel `tft_weights_hourly`.
- Bila `"direct"` dihapus dari `SEED_STRATEGIES`, direct memakai satu run
  produksi dari perbandingan di atas, sehingga simpangan bakunya tidak bisa
  dihitung.
- Untuk tiap pasangan strategi dan tiap (stasiun, horizon) dilaporkan selisih
  rata-rata NSE antar-seed (Δ = a − b) dan standard error (SE) selisih itu.
  Selisih dianggap melebihi noise bila |Δ| > 2 × SE. Dengan 3 seed ini hanya
  indikasi kasar; tambah seed untuk kesimpulan yang lebih kuat.
- Metrik interval `mimo_q` (PICP, PICP_high, MPIW, pinball) dilaporkan sebagai
  rata-rata ± std antar-seed per horizon.
- Bila `SAVE_SEEDS_TO_BQ = True`, metrik per seed disimpan ke
  `tft_seed_metrics_hourly` dan bobot tiap model MIMO per (seed, stasiun) ke
  `tft_mimo_seed_weights_hourly` (kolom `model` membedakan `tft_mimo` dan
  `tft_mimo_q`). Bobot direct multi-seed tidak disimpan.
- Di GPU, seed yang sama belum tentu memberi hasil yang identik bit per bit,
  karena kernel cuDNN untuk RNN tidak sepenuhnya deterministik. Variasi kecil
  ini ikut terukur sebagai bagian dari variasi antar-run.

Perkiraan waktu: TFT jauh lebih lambat daripada LSTM/GRU, karena VSN menjalankan
GRN terpisah untuk tiap fitur di tiap timestep. Cell setup di bawah mencetak
perkiraan dari waktu training MIMO di atas. Perhatikan batas durasi sesi Colab;
kurangi `SEEDS` atau hapus `"direct"` dari `SEED_STRATEGIES` bila terlalu lama.

In [ ]:
import random

SEEDS            = [0, 1, 2]
SEED_STRATEGIES  = ["direct", "mimo", "mimo_q"]   # hapus "direct" = pakai satu run direct produksi
SAVE_SEEDS_TO_BQ = True    # True = simpan metrik per seed ke {MODEL_NAME}_seed_metrics_hourly
                           #        + bobot MIMO per seed ke {MIMO_MODEL_NAME}_seed_weights_hourly


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)   # juga men-seed semua GPU


def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


def seed_metric_rows(strategy, seed, station, h, metrics):
    return [{"strategy": strategy, "seed": int(seed), "station": station, "horizon": int(h),
             "metric": m, "value": float(v)} for m, v in metrics.items()]


n_fits = len(SEEDS) * len(station_cols) * (
    ("mimo" in SEED_STRATEGIES) + ("mimo_q" in SEED_STRATEGIES)
    + len(HORIZONS) * ("direct" in SEED_STRATEGIES))
print(f"Seeds: {SEEDS} | SEED_STRATEGIES={SEED_STRATEGIES} | model yang dilatih: {n_fits}")

if globals().get("mimo_train_info") and globals().get("mimo_q_train_info"):
    t_mimo = sum(i["seconds"] for i in mimo_train_info.values()) / 60
    t_q    = sum(i["seconds"] for i in mimo_q_train_info.values()) / 60
    per_seed = (t_mimo * ("mimo" in SEED_STRATEGIES) + t_q * ("mimo_q" in SEED_STRATEGIES)
                + len(HORIZONS) * t_mimo * ("direct" in SEED_STRATEGIES))
    print(f"Perkiraan waktu: ~{per_seed * len(SEEDS):.0f} menit "
          f"(dari waktu training MIMO di atas; direct diasumsikan ~{len(HORIZONS)}x MIMO)")

In [ ]:
# --- Training + evaluasi test per seed ------------------------------------------
# Dataset dibangun sekali per stasiun; urutan batch tetap dikendalikan seed.
# Model direct dievaluasi lalu dibuang; bobot MIMO ikut diserialisasi bila SAVE_SEEDS_TO_BQ.
seed_rows = []
mimo_seed_weights = []   # bobot MIMO per (model, seed, stasiun), diunggah di cell simpan BQ
t_start = time.perf_counter()

for target in station_cols:
    feats = features_for(target)
    mimo_tr, mimo_va, mimo_te = (TMADatasetMIMO(df, target, feats, LOOKBACK, HORIZONS)
                                 for df in (df_train, df_val, df_test))
    n = len(mimo_te)
    direct_sets = ({h: [TMADataset(df, target, feats, LOOKBACK, h) for df in (df_train, df_val, df_test)]
                    for h in HORIZONS} if "direct" in SEED_STRATEGIES else {})

    for seed in SEEDS:
        y_ref = None   # target test MIMO, untuk cek kesejajaran dengan direct
        for strategy in ("mimo", "mimo_q"):
            if strategy not in SEED_STRATEGIES:
                continue
            quantiles = QUANTILES if strategy == "mimo_q" else None
            set_seed(seed)
            model, _ = fit(make_tft(target, multi=True, quantiles=quantiles),
                           make_loader(mimo_tr, True), make_loader(mimo_va, False),
                           pinball_loss if quantiles else mse_loss)
            test_loader = make_loader(mimo_te, False)
            if quantiles:
                qp, yt, crossing = get_quantile_predictions(model, test_loader)
                for j, h in enumerate(HORIZONS):
                    metrics = {**compute_metrics(yt[:, j], qp[:, j, Q_MEDIAN]),
                               **interval_metrics(yt[:, j], qp[:, j], QUANTILES),
                               "crossing": float(crossing[j] * 100)}
                    seed_rows += seed_metric_rows(strategy, seed, target, h, metrics)
            else:
                yp, yt = get_predictions(model, test_loader)
                for j, h in enumerate(HORIZONS):
                    seed_rows += seed_metric_rows(strategy, seed, target, h,
                                                  compute_metrics(yt[:, j], yp[:, j]))
            y_ref = yt

            if SAVE_SEEDS_TO_BQ:
                buf = io.BytesIO()
                torch.save(model.state_dict(), buf)
                mimo_seed_weights.append({
                    "model": MIMO_Q_MODEL_NAME if quantiles else MIMO_MODEL_NAME,
                    "seed": int(seed), "station": target,
                    "weights": buf.getvalue(), "n_bytes": buf.getbuffer().nbytes,
                })

        for j, h in enumerate(HORIZONS):
            if h not in direct_sets:
                continue
            tr, va, te = direct_sets[h]
            set_seed(seed)
            model, _ = fit(make_tft(target), make_loader(tr, True), make_loader(va, False), mse_loss)
            yp, yt = get_predictions(model, make_loader(te, False))
            yp, yt = yp[:n], yt[:n]   # window test yang sama dengan MIMO
            if y_ref is not None:
                assert np.allclose(yt, y_ref[:, j], equal_nan=True), f"target tidak sejajar ({target}, h+{h})"
            seed_rows += seed_metric_rows("direct", seed, target, h, compute_metrics(yt, yp))

    print(f"{target:<20} selesai | {(time.perf_counter() - t_start) / 60:.1f} menit berjalan")

if "direct" not in SEED_STRATEGIES:   # pembanding: satu run direct produksi (dari perbandingan di atas)
    seed_rows += comparison_df[comparison_df.strategy == "direct"].assign(seed=-1).to_dict("records")

seeds_df = pd.DataFrame(seed_rows)
print(f"\nseeds_df: {len(seeds_df)} baris")

In [ ]:
# --- Ringkasan multi-seed -------------------------------------------------------
from scipy.stats import wilcoxon

nse_s = seeds_df[seeds_df.metric == "NSE"]
stats = (nse_s.groupby(["station", "horizon", "strategy"])["value"]
         .agg(["mean", "std", "count"]).unstack("strategy"))
seed_strategies = [s for s in STRATEGIES if s in stats.columns.get_level_values("strategy")]

seed_tbl = pd.DataFrame({f"{s}_{k}": stats[(k, s)] for s in seed_strategies for k in ("mean", "std")})
print(f"NSE test, rata-rata ± std atas {len(SEEDS)} seed:")
display(seed_tbl.round(4))

print("\nVariasi antar-seed: median std NSE per horizon")
display(seed_tbl.groupby(level="horizon")[[f"{s}_std" for s in seed_strategies]].median().round(4))

# Selisih per pasangan; SE selisih rata-rata (std NaN, mis. direct satu run, dianggap 0).
pair_tbls = {}
for a, b in PAIRS:
    if a not in seed_strategies or b not in seed_strategies:
        continue
    delta = stats[("mean", a)] - stats[("mean", b)]
    se = np.sqrt(stats[("std", a)].fillna(0) ** 2 / stats[("count", a)]
                 + stats[("std", b)].fillna(0) ** 2 / stats[("count", b)])
    t = pd.DataFrame({"delta": delta, "se": se})
    t["lebih_baik"] = np.where(t["delta"].abs() > 2 * t["se"], np.where(t["delta"] > 0, a, b), "-")
    pair_tbls[(a, b)] = t

    print(f"\n{a} vs {b} (delta = {a} - {b}; di luar noise: |delta| > 2*SE):")
    for h in HORIZONS:
        d = t.xs(h, level="horizon")
        print(f"  h+{h:<2}: {a} menang {int((d['delta'] > 0).sum())}/{len(d)} | "
              f"median delta={d['delta'].median():+.4f} | mean delta={d['delta'].mean():+.4f} | "
              f"di luar noise: {a} {int((d['lebih_baik'] == a).sum())}, "
              f"{b} {int((d['lebih_baik'] == b).sum())} | Wilcoxon p={wilcoxon(d['delta']).pvalue:.3f}")

# Interval mimo_q: rata-rata atas stasiun per seed, lalu mean ± std antar-seed.
if "mimo_q" in seed_strategies:
    iv_s = seeds_df[(seeds_df.strategy == "mimo_q")
                    & seeds_df.metric.isin(["PICP", "PICP_high", "MPIW", "pinball", "crossing"])]
    per_seed = iv_s.groupby(["horizon", "metric", "seed"])["value"].mean()
    print(f"\nPrediction interval {NOMINAL_PI}% (mimo_q), rata-rata {len(station_cols)} stasiun, "
          f"mean ± std antar-seed:")
    display(per_seed.groupby(["horizon", "metric"]).agg(["mean", "std"]).unstack("metric").round(4))

In [ ]:
# --- Plot: selisih NSE per pasangan strategi per stasiun, error bar = 2 x SE ----
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(pair_tbls), len(HORIZONS),
                         figsize=(5 * len(HORIZONS), (0.45 * len(station_cols) + 1.2) * len(pair_tbls)),
                         sharey=True, squeeze=False)
for r, ((a, b), t) in enumerate(pair_tbls.items()):
    side_colors = {a: S_COLORS[a], b: S_COLORS[b], "-": "#bdbdbd"}
    for c, h in enumerate(HORIZONS):
        ax = axes[r, c]
        d = t.xs(h, level="horizon").reindex(station_cols[::-1])
        y = np.arange(len(d))
        for yi, (_, row) in zip(y, d.iterrows()):
            ax.errorbar(row["delta"], yi, xerr=2 * row["se"], fmt="o", ms=6, capsize=3,
                        color=side_colors[row["lebih_baik"]])
        ax.axvline(0, color="k", lw=0.8, ls="--")
        ax.set_yticks(y, d.index, fontsize=8)
        ax.set_title(f"{a} − {b}  |  h+{h}", fontsize=10)
        if r == len(pair_tbls) - 1:
            ax.set_xlabel("Δ NSE")

handles = [plt.Line2D([], [], color=S_COLORS[s], marker="o", ls="", label=f"{s} lebih baik")
           for s in seed_strategies]
handles.append(plt.Line2D([], [], color="#bdbdbd", marker="o", ls="", label="dalam noise"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.suptitle(f"{MODEL_NAME.upper()}: selisih NSE antar strategi atas {len(SEEDS)} seed (± 2 SE)")
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

In [ ]:
# --- (Opsional) simpan metrik + bobot MIMO per seed ke BigQuery, tabel terpisah -
if SAVE_SEEDS_TO_BQ:
    _seed_schema = [
        bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("strategy", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("seed", "INTEGER", mode="REQUIRED"),   # -1 = run direct produksi
        bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
    ]
    _seeds = seeds_df.assign(model=MODEL_NAME)[[f.name for f in _seed_schema]]
    load_df(_seeds, f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_seed_metrics_hourly", _seed_schema)

    # Bobot MIMO per seed (kolom model: tft_mimo / tft_mimo_q). Memuat ulang satu model:
    #   m = make_tft(station, multi=True, quantiles=QUANTILES if model == MIMO_Q_MODEL_NAME else None)
    #   m.load_state_dict(torch.load(io.BytesIO(blob), map_location=DEVICE))
    _sw_schema = [
        bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("seed", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
        bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
    ]
    _sw = pd.DataFrame(globals().get("mimo_seed_weights", []), columns=[f.name for f in _sw_schema[:-1]])
    if _sw.empty:
        print("Bobot MIMO per seed kosong: jalankan ulang cell training multi-seed dengan SAVE_SEEDS_TO_BQ = True.")
    else:
        _sw["saved_at"] = pd.Timestamp.now(tz="UTC").tz_localize(None)
        load_df(_sw, f"{PROJECT_ID}.{DATASET}.{MIMO_MODEL_NAME}_seed_weights_hourly", _sw_schema)
    print(f"\n✔ Metrik multi-seed '{MODEL_NAME}' tersimpan di BigQuery.")
else:
    print("SAVE_SEEDS_TO_BQ = False: metrik multi-seed tidak ditulis ke BigQuery.")